In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Question 1

In [11]:
from embedder import Embedder

embedder = Embedder()
v = embedder.encode("How does approximate nearest neighbor search work?")

print (len(v))
print(v[0])

384
-0.02058203437252893


# Question 2

In [12]:
target = next(d for d in documents if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md" )

v_target = embedder.encode(target["content"])

similarity = v.dot(v_target)
print(similarity)

0.36107027225589694


# Question 3

In [14]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [16]:
chunks_contents = [c["content"] for c in chunks]

embedded_chunks = embedder.encode_batch(chunks_contents)

scores = embedded_chunks.dot(v)
best_idx  = scores.argmax()

print(chunks[best_idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


# Question 4

In [41]:
from minsearch import VectorSearch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Completely replace search with no reference to original [as it has a bug]
def fixed_search(self, query_vector, filter_dict=None, num_results=10, output_ids=False):
    if len(self.docs) == 0 or self.vectors is None:
        return []
    query_vector_2d = query_vector.reshape(1, -1)
    scores = cosine_similarity(query_vector_2d, self.vectors).flatten()
    top_indices = scores.argsort()[::-1][:num_results]
    return [self.docs[i] for i in top_indices]

VectorSearch.search = fixed_search

vector_search = VectorSearch()
vector_search.fit(embedded_chunks, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = vector_search.search(query_vector, num_results=5)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


# Question 5

In [42]:
from minsearch import Index

idx = Index(text_fields=["content"], keyword_fields=["filename"])
idx.fit(chunks)

In [43]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embedder.encode(query)

vector search

In [45]:
vector_results = vector_search.search(query_vector, num_results=5)
vector_filenames = set(r["filename"] for r in vector_results)

keyword search

In [46]:
text_results = idx.search(query, num_results=5)
text_filenames = set(r["filename"] for r in text_results)

difference between both

In [47]:
print(vector_filenames - text_filenames)

{'02-vector-search/lessons/08-pgvector.md'}


# Question 6

In [48]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [49]:
query = "How do I give the model access to tools?"
query_vector = embedder.encode(query)

In [50]:
vector_results = vector_search.search(query_vector, num_results=5)
text_results = idx.search(query, num_results=5)

final = rrf([vector_results, text_results])
print(final[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
